# Tamr Vision: run the complete system

SDAIA Academy, Computer Vision Systems Development (SDA-AIE-212).

Run the cells top to bottom (click a cell, then press Shift+Enter). Total time on a free Colab CPU: about 5 minutes.

## Step 1. Upload the package
Run this cell, click **Choose Files**, and pick `tamr_demo.zip` from your computer.

In [ ]:
from google.colab import files
up = files.upload()
import zipfile, os
zipfile.ZipFile('tamr_demo.zip').extractall('.')
os.chdir('tamr_demo')
print('unpacked into', os.getcwd())
print(sorted(os.listdir('.')))

## Step 2. Install the libraries
About 2 minutes. Ignore the warnings; only a red 'ERROR' line matters.

In [ ]:
!pip -q install ultralytics onnx onnxruntime onnxslim opencv-python-headless pyarrow 2>&1 | grep -iE 'error|Successfully' || echo 'done'
import torch, ultralytics, onnxruntime; print('torch', torch.__version__, '| ultralytics', ultralytics.__version__, '| onnxruntime', onnxruntime.__version__)

## Step 3. Run the system end to end
The script generates the dataset, exports the classifier and the detector to ONNX, proves parity, runs the edge loop on 80 test frames using only `config/grading_rules.json`, and writes `report.html`.

Watch for the five `==== Step` lines and the word `DONE` at the end.

In [ ]:
!python run_demo.py 2>&1 | grep -vE 'Warning|WARNING|warn'

## Step 4. Open the report

In [ ]:
from IPython.display import HTML, display
display(HTML(open('report.html').read()))

## Step 5. Look at one decision log line
This is what the line controller receives for every frame.

In [ ]:
import json
logs = json.load(open('runs/decision_logs.json'))
print(json.dumps(logs[1], indent=1))

## Step 6. Change the rules and run again
The decision is not in the code. Edit one number in the config, run the system again, and watch the decisions move.

Below we lower `area_reject_threshold_pct` from 8 to 3 (any defect area above 3 percent becomes a reject).

In [ ]:
import json
rules = json.load(open('config/grading_rules.json'))
rules['area_reject_threshold_pct'] = 3.0
rules['rules_version'] = 'grading-rules-v1-strict'
json.dump(rules, open('config/grading_rules.json', 'w'), indent=2)
!python run_demo.py 2>&1 | grep -E '^decisions|^accuracy|DONE'

Compare the `decisions:` line with the one from Step 3. More rejects, lower accuracy against the plant's official grades: a stricter rule is a business choice, and the log stamps which rules version made each decision.

To put the original rules back, run the cell below.

In [ ]:
rules['area_reject_threshold_pct'] = 8.0
rules['rules_version'] = 'grading-rules-v1'
json.dump(rules, open('config/grading_rules.json', 'w'), indent=2)
print('rules restored')

## Step 7. Download the results
The report and the ONNX models.

In [ ]:
from google.colab import files
files.download('report.html')
files.download('dist/grade_resnet18_int8.onnx')
files.download('runs/decision_logs.json')